# 10 — Schema-Driven Build of the COSMoS Graph

First step in the cosmos-graph pipeline — converts the CDISC Excel exports into a multi-sheet interim that preserves the authored graph shape at VLM-row grain. No external-data lookup, no CT resolution, no NCIt enrichment. Schema-driven via `linkml-runtime.SchemaView`.

**Input**
- `cosmos-bc-dss/downloads/cdisc_sdtm_dataset_specializations_latest.xlsx` — VLM-row grain
- `cosmos-bc-dss/downloads/cdisc_biomedical_concepts_latest.xlsx` — BC identity + hierarchy
- `cosmos-graph/reference/cosmos_linkml/cosmos_sdtm_model.yaml` — LinkML schema for SDTMGroup / SDTMVariable / RelationShip

**Output**
- `interim/COSMoS_Graph.xlsx` — 11 sheets:
  - `ReadMe` — provenance and sheet inventory
  - BC layer: `BC`, `BC_Parents`, `BC_Categories`, `Categories`, `Coding`, `DataElementConcepts`
  - SDTM layer: `DSS`, `Variables`, `Relationships`, `Codelists`

**Out of scope for this notebook** — CT resolution (→ notebook 20), validation (→ notebook 30).

Design rationale: [`../docs/COSMoS_Graph.md`](../docs/COSMoS_Graph.md).

## 1. Imports


In [1]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd
from linkml_runtime.utils.schemaview import SchemaView

print(f'python : {sys.version.split()[0]}')
print(f'pandas : {pd.__version__}')


python : 3.10.12
pandas : 2.3.3


/sessions/gifted-eloquent-faraday/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration


In [2]:
REPO_ROOT = Path.cwd().parent.parent  # notebooks/ -> cosmos-graph/ -> repo root

DOWNLOADS = REPO_ROOT / 'cosmos-bc-dss' / 'downloads'
INTERIM   = REPO_ROOT / 'cosmos-graph' / 'interim'
SCHEMAS   = REPO_ROOT / 'cosmos-graph' / 'reference' / 'cosmos_linkml'

DSS_XLSX    = DOWNLOADS / 'cdisc_sdtm_dataset_specializations_latest.xlsx'
BC_XLSX     = DOWNLOADS / 'cdisc_biomedical_concepts_latest.xlsx'
SDTM_SCHEMA = SCHEMAS / 'cosmos_sdtm_model.yaml'
OUT_XLSX    = INTERIM / 'COSMoS_Graph.xlsx'

for p in (DSS_XLSX, BC_XLSX, SDTM_SCHEMA):
    assert p.exists(), f'missing: {p}'
INTERIM.mkdir(parents=True, exist_ok=True)

print(f'DSS_XLSX : {DSS_XLSX.relative_to(REPO_ROOT)}')
print(f'BC_XLSX  : {BC_XLSX.relative_to(REPO_ROOT)}')
print(f'out      : {OUT_XLSX.relative_to(REPO_ROOT)}')


DSS_XLSX : cosmos-bc-dss/downloads/cdisc_sdtm_dataset_specializations_latest.xlsx
BC_XLSX  : cosmos-bc-dss/downloads/cdisc_biomedical_concepts_latest.xlsx
out      : cosmos-graph/interim/COSMoS_Graph.xlsx


## 3. Load LinkML schemas

The schemas are the authoritative model — we use SchemaView to enumerate slots and drive column selection. This notebook does not attempt structural validation of the xlsx against the schema — that's notebook 30's job. We rely on the schema as a slot catalogue.


In [3]:
sv_sdtm = SchemaView(str(SDTM_SCHEMA))

sdtm_classes = list(sv_sdtm.all_classes())
print(f'sdtm schema: {len(sdtm_classes)} classes')

sdtm_var_slots = sv_sdtm.class_induced_slots('SDTMVariable')
print(f'\nSDTMVariable: {len(sdtm_var_slots)} induced slots')
print('  ' + ', '.join(s.name for s in sdtm_var_slots[:12]) + ' ...')


sdtm schema: 7 classes

SDTMVariable: 19 induced slots
  name, dataElementConceptId, isNonStandard, codelist, subsetCodelist, valueList, assignedTerm, role, relationship, dataType, length, format ...


## 4. Read source xlsx


In [4]:
dss_df = pd.read_excel(DSS_XLSX, sheet_name='SDTM Dataset Specializations')
print(f'DSS xlsx (VLM-row grain): {dss_df.shape}')

bc_concepts   = pd.read_excel(BC_XLSX, sheet_name='Biomedical Concepts')
bc_hierarchy  = pd.read_excel(BC_XLSX, sheet_name='BC Hierarchy')
categories_df = pd.read_excel(BC_XLSX, sheet_name='Categories')
print(f'BC Biomedical Concepts : {bc_concepts.shape}  (one row per BC per external-code mapping)')
print(f'BC Hierarchy           : {bc_hierarchy.shape}  (one row per BC)')
print(f'Categories             : {categories_df.shape}  (one row per category token)')


DSS xlsx (VLM-row grain): (12677, 32)


BC Biomedical Concepts : (5743, 17)  (one row per BC per external-code mapping)
BC Hierarchy           : (1345, 11)  (one row per BC)
Categories             : (385, 1)  (one row per category token)


## 5. Column rename — xlsx surface → canonical snake_case

The xlsx uses CDISC's publication names (`vlm_group_id`, `vlm_source`, `bc_id`, etc.). We rename to the canonical snake_case vocabulary used across `interim/COSMoS_Graph.xlsx` sheets. The rename pairs correspond one-to-one to LinkML slot names in snake_case form; see [`../docs/COSMoS_Graph.md`](../docs/COSMoS_Graph.md) §5 for the full xlsx ↔ LinkML table.


In [5]:
DSS_RENAME = {
    'package_date':               'package_date',
    'bc_id':                      'bc_id',
    'sdtmig_start_version':       'sdtmig_start_version',
    'sdtmig_end_version':         'sdtmig_end_version',
    'domain':                     'domain',
    'vlm_source':                 'source',
    'vlm_group_id':               'ds_id',
    'short_name':                 'ds_short_name',
    'sdtm_variable':              'variable_name',
    'dec_id':                     'dec_id',
    'nsv_flag':                   'is_nonstandard',
    'codelist':                   'codelist_concept_id',
    'codelist_submission_value':  'codelist_submission_value',
    'subset_codelist':            'subset_codelist',
    'value_list':                 'value_list',
    'assigned_term':              'assigned_term_concept_id',
    'assigned_value':             'assigned_term_value',
    'role':                       'role',
    'subject':                    'subject',
    'linking_phrase':             'linking_phrase',
    'predicate_term':             'predicate_term',
    'object':                     'object',
    'data_type':                  'data_type',
    'length':                     'length',
    'format':                     'format',
    'significant_digits':         'significant_digits',
    'mandatory_variable':         'mandatory_variable',
    'mandatory_value':            'mandatory_value',
    'origin_type':                'origin_type',
    'origin_source':              'origin_source',
    'comparator':                 'comparator',
    'vlm_target':                 'vlm_target',
}

missing = set(dss_df.columns) - set(DSS_RENAME.keys())
extra   = set(DSS_RENAME.keys()) - set(dss_df.columns)
assert not missing and not extra, f'DSS_RENAME mismatch. missing={missing}, extra={extra}'

df = dss_df.rename(columns=DSS_RENAME).copy()
print(f'renamed: {df.shape}')
print(f'columns: {list(df.columns)}')


renamed: (12677, 32)
columns: ['package_date', 'bc_id', 'sdtmig_start_version', 'sdtmig_end_version', 'domain', 'source', 'ds_id', 'ds_short_name', 'variable_name', 'dec_id', 'is_nonstandard', 'codelist_concept_id', 'codelist_submission_value', 'subset_codelist', 'value_list', 'assigned_term_concept_id', 'assigned_term_value', 'role', 'subject', 'linking_phrase', 'predicate_term', 'object', 'data_type', 'length', 'format', 'significant_digits', 'mandatory_variable', 'mandatory_value', 'origin_type', 'origin_source', 'comparator', 'vlm_target']


## 6. Build `DSS` sheet

One row per Dataset Specialization. DSS-level facts only — fields that are constant across all variable rows of a given `ds_id`. Per-variable facts live in `Variables`.


In [6]:
DSS_COLS = ['ds_id', 'bc_id', 'domain', 'source', 'ds_short_name',
            'sdtmig_start_version', 'sdtmig_end_version', 'package_date']

dss_sheet = df[DSS_COLS].drop_duplicates(subset=['ds_id']).reset_index(drop=True)
assert dss_sheet['ds_id'].is_unique, 'ds_id not unique after dedup'

# Integrity check — no DSS-level field should vary within a ds_id group.
per_dss_variance = df.groupby('ds_id')[DSS_COLS].nunique().max()
varying = per_dss_variance[per_dss_variance > 1]
assert varying.empty, f'DSS-level columns vary within ds_id: {varying.to_dict()}'

print(f'DSS sheet: {dss_sheet.shape}')
print(f'distinct domains: {dss_sheet.domain.nunique()}')
dss_sheet.head()


DSS sheet: (1326, 8)
distinct domains: 32


,ds_id,bc_id,domain,source,ds_short_name,sdtmig_start_version,sdtmig_end_version,package_date
0,AE,C179175,AE,AE.AETERM,Adverse Event Free Text Format,3-4,NaN,2025-07-01
1,AEPRESP,C179175,AE,AE.AETERM,Adverse Event Prespecified,3-4,NaN,2025-07-01
2,COLLECTING,C70945,BE,BE.BETERM,Biospecimen Collection,3-4,NaN,2023-07-06
3,CULTURING,C17201,BE,BE.BETERM,Biospecimen Culturing,3-4,NaN,2023-07-06
4,FLASHFREEZING,C63521,BE,BE.BETERM,Biospecimen Flash Freezing,3-4,NaN,2023-07-06


## 7. Build `Variables` sheet

One row per SDTMVariable within a DSS — VLM-row grain. DSS-level columns dropped (they're in the `DSS` sheet, joined via `ds_id`). `bc_id` is kept as a denormalised join key to `BC` for convenience.


In [7]:
VARIABLES_COLS = [
    'ds_id', 'bc_id',                                              # join keys
    'variable_name', 'role', 'is_nonstandard', 'dec_id',           # variable identity
    'data_type', 'length', 'format', 'significant_digits',          # data shape
    'mandatory_variable', 'mandatory_value', 'comparator',          # pin vs qualifier
    'origin_type', 'origin_source', 'vlm_target',                   # provenance
    'assigned_term_concept_id', 'assigned_term_value',              # pinned term
    'codelist_concept_id', 'codelist_submission_value',             # codelist binding
    'subset_codelist', 'value_list',                                # constrained values
    'subject', 'linking_phrase', 'predicate_term', 'object',        # reification quad
]

variables_sheet = df[VARIABLES_COLS].reset_index(drop=True)
print(f'Variables sheet: {variables_sheet.shape}')
print(f'role distribution: {dict(variables_sheet.role.value_counts())}')
variables_sheet.head()


Variables sheet: (12677, 26)
role distribution: {'Qualifier': np.int64(9867), 'Timing': np.int64(1388), 'Topic': np.int64(1334), 'Identifier': np.int64(81)}


,ds_id,bc_id,variable_name,role,is_nonstandard,dec_id,data_type,length,format,significant_digits,...,assigned_term_concept_id,assigned_term_value,codelist_concept_id,codelist_submission_value,subset_codelist,value_list,subject,linking_phrase,predicate_term,object
0,AE,C179175,AETERM,Topic,N,C78541,text,200.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,AETERM,is original text for,IS_ORIGINAL_TEXT_FOR,AEDECODE
1,AE,C179175,AEDECOD,Qualifier,N,C83344,text,200.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,AEDECOD,is a dictionary-derived term for the value in,IS_DERIVED_FROM,AETERM
2,AE,C179175,AELLT,Qualifier,N,NaN,text,200.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,AELLT,is a dictionary-derived term for the value in,IS_DERIVED_FROM,AETERM
3,AE,C179175,AELLTCD,Qualifier,N,NaN,integer,8.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,AELLTCD,is the code for the value in,IS_DECODED_BY,AETERM
4,AE,C179175,AEPTCD,Qualifier,N,NaN,integer,8.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,AEPTCD,is the code for the value in,IS_DECODED_BY,AETERM


## 8. Build `Relationships` sheet

Long-format reification quads. One row per reified edge. Variable rows without a populated quad are excluded — tracked as an anomaly.


In [8]:
rel_mask = (
    variables_sheet['subject'].notna()
    & variables_sheet['linking_phrase'].notna()
    & variables_sheet['predicate_term'].notna()
    & variables_sheet['object'].notna()
)
relationships_sheet = variables_sheet.loc[rel_mask, [
    'ds_id', 'variable_name', 'subject', 'linking_phrase', 'predicate_term', 'object',
]].reset_index(drop=True)

dropped = (~rel_mask).sum()
print(f'Relationships sheet: {relationships_sheet.shape}')
print(f'dropped {dropped} variable rows with empty reification quad')
relationships_sheet.head()


Relationships sheet: (12364, 6)
dropped 313 variable rows with empty reification quad


,ds_id,variable_name,subject,linking_phrase,predicate_term,object
0,AE,AETERM,AETERM,is original text for,IS_ORIGINAL_TEXT_FOR,AEDECODE
1,AE,AEDECOD,AEDECOD,is a dictionary-derived term for the value in,IS_DERIVED_FROM,AETERM
2,AE,AELLT,AELLT,is a dictionary-derived term for the value in,IS_DERIVED_FROM,AETERM
3,AE,AELLTCD,AELLTCD,is the code for the value in,IS_DECODED_BY,AETERM
4,AE,AEPTCD,AEPTCD,is the code for the value in,IS_DECODED_BY,AETERM


## 9. Build `Codelists` sheet

Deduped codelist bindings with usage counts. One row per `(codelist_concept_id, codelist_submission_value)` pair actually referenced by any variable. `subset_codelist` is out of scope for this sheet — it stays in `Variables` as authored (range: string).


In [9]:
codelists_sheet = (
    variables_sheet[['codelist_concept_id', 'codelist_submission_value']]
    .dropna(subset=['codelist_concept_id'])
    .value_counts(dropna=False)
    .reset_index(name='variable_uses_count')
    .sort_values('codelist_concept_id')
    .reset_index(drop=True)
)
print(f'Codelists sheet: {codelists_sheet.shape}')
codelists_sheet.head(10)


Codelists sheet: (291, 3)


,codelist_concept_id,codelist_submission_value,variable_uses_count
0,C100129,QSCAT,17
1,C100131,ADCTN,16
2,C100132,ADCTC,16
3,C100133,BPRSA1TN,18
4,C100134,BPRSA1TC,18
5,C100169,KPSS01TN,1
6,C100170,KPSS01TC,1
7,C101805,AIMS01TC,12
8,C101806,AIMS01TN,12
9,C101815,ECOG1TC,1


## 10. Build `BC` sheet

One row per Biomedical Concept — authored identity, hierarchy info, plus our `bc_type` classification:

- `full` — BC is in the Biomedical Concepts sheet and has at least one DSS.
- `full_no_ds` — BC is in Biomedical Concepts but no DSS has been authored (gap marker).
- `hierarchy_only` — BC is only in the BC Hierarchy sheet (structural parent, not a standalone concept).

`ncit_code` is joined from the Biomedical Concepts sheet; `bc_parent_label` is derived by mapping `parent_bc_id` back to the BC's `bc_short_name`.


In [10]:
# BC identity — start from BC Hierarchy (authoritative BC list, one row per BC)
bc_base = bc_hierarchy[[
    'bc_id', 'short_name', 'parent_bc_id', 'bc_categories', 'synonyms',
    'result_scales', 'definition', 'bc_hierarchy_full',
]].rename(columns={
    'short_name':        'bc_short_name',
    'definition':        'bc_definition',
    'synonyms':          'bc_synonyms',
    'bc_hierarchy_full': 'bc_hierarchy_path',
})

# Join ncit_code from Biomedical Concepts (first row per bc_id — ncit_code is stable within a BC)
bc_ncit = (
    bc_concepts[['bc_id', 'ncit_code']]
    .dropna(subset=['bc_id'])
    .drop_duplicates(subset=['bc_id'])
)
bc_sheet = bc_base.merge(bc_ncit, on='bc_id', how='left')

# Parent label — map parent_bc_id -> bc_short_name
name_map = dict(zip(bc_sheet['bc_id'], bc_sheet['bc_short_name']))
bc_sheet['bc_parent_label'] = bc_sheet['parent_bc_id'].map(name_map)

# bc_type classification
bcs_in_concepts = set(bc_concepts['bc_id'].dropna().unique())
bcs_with_dss    = set(dss_sheet['bc_id'].dropna().unique())

def classify_bc(bc_id):
    if bc_id not in bcs_in_concepts:
        return 'hierarchy_only'
    if bc_id in bcs_with_dss:
        return 'full'
    return 'full_no_ds'

bc_sheet['bc_type'] = bc_sheet['bc_id'].apply(classify_bc)

BC_COLS = [
    'bc_id', 'ncit_code', 'bc_short_name', 'bc_definition', 'bc_synonyms',
    'bc_categories', 'bc_hierarchy_path', 'bc_parent_label', 'bc_type',
    'result_scales',
]
bc_sheet = bc_sheet[BC_COLS].reset_index(drop=True)

print(f'BC sheet: {bc_sheet.shape}')
print(f'bc_type distribution:')
print(bc_sheet['bc_type'].value_counts())
bc_sheet.head()


BC sheet: (1345, 10)
bc_type distribution:
bc_type
full          903
full_no_ds    442
Name: count, dtype: int64


,bc_id,ncit_code,bc_short_name,bc_definition,bc_synonyms,bc_categories,bc_hierarchy_path,bc_parent_label,bc_type,result_scales
0,C127127,C127127,4-(Methylnitrosamino)-1-(3-Pyridyl)-1-Butanol,The major metabolite of the potentially carcin...,NNAL;Nicotine-Derived Nitrosamine Alcohol,Laboratory Tests;Nitroso Carcinogens;Carcinoge...,Carcinogenic Nitrosamine (C45398); 4-(Methylni...,Carcinogenic Nitrosamine,full_no_ds,Quantitative
1,C115789,C115789,6 Minute Walk Functional Test,A standardized rating scale developed by Bruno...,6MWT;SIXMW1;SIX MINUTE WALK,QRS;Functional Assessment;6 Minute Walk Functi...,Functional Assessment (C81250); 6 Minute Walk ...,Functional Assessment,full_no_ds,NaN
2,C115800,C115800,6MWT - Distance at 1 Minute,6 Minute Walk Test (6MWT) Distance at 1 minute.,SIXMW101;SIXMW1-Distance at 1 Minute,QRS;Functional Assessment;6 Minute Walk Functi...,Clinical or Research Assessment Question (C911...,6MWT Functional Test Question,full,Quantitative
3,C115801,C115801,6MWT - Distance at 2 Minutes,6 Minute Walk Test (6MWT) Distance at 2 minutes.,SIXMW102;SIXMW1-Distance at 2 Minutes,QRS;Functional Assessment;6 Minute Walk Functi...,Clinical or Research Assessment Question (C911...,6MWT Functional Test Question,full,Quantitative
4,C115802,C115802,6MWT - Distance at 3 Minutes,6 Minute Walk Test (6MWT) Distance at 3 minutes.,SIXMW103;SIXMW1-Distance at 3 Minutes,QRS;Functional Assessment;6 Minute Walk Functi...,Clinical or Research Assessment Question (C911...,6MWT Functional Test Question,full,Quantitative


## 10a. Build `Coding` sheet

One row per `(bc_id × external code system × code)`. Projects the inlined `Coding` list from the LinkML `BiomedicalConcept` schema. Source columns `system`, `system_name`, `code` come from the xlsx `Biomedical Concepts` sheet where inlined children are cross-joined onto BC rows.

In [11]:
coding_sheet = (
    bc_concepts[['bc_id', 'system', 'system_name', 'code']]
    .dropna(subset=['bc_id'])
    .drop_duplicates()
    .reset_index(drop=True)
)
# Drop rows with no external-code content (arise from the DEC-only cross-join rows)
coding_sheet = coding_sheet[
    coding_sheet[['system', 'system_name', 'code']].notna().any(axis=1)
].reset_index(drop=True)

print(f'Coding sheet: {coding_sheet.shape}')
print(f'BCs with at least one coding: {coding_sheet["bc_id"].nunique()}')
print(f'distinct systems: {coding_sheet["system"].nunique()}')
print(coding_sheet['system'].value_counts().head(10))
coding_sheet.head()

Coding sheet: (105, 4)
BCs with at least one coding: 105
distinct systems: 2
system
http://loinc.org/    104
https://loinc.org      1
Name: count, dtype: int64


,bc_id,system,system_name,code
0,C115805,http://loinc.org/,LOINC,64098-7
1,C121113,http://loinc.org/,LOINC,9264-3
2,C103346,http://loinc.org/,LOINC,8355-0
3,C117773,http://loinc.org/,LOINC,8625-6
4,C117779,http://loinc.org/,LOINC,8633-0


## 10b. Build `DataElementConcepts` sheet

One row per `(bc_id × DEC)`. Projects the inlined `DataElementConcept` list from the LinkML `BiomedicalConcept` schema. Columns come from the xlsx `Biomedical Concepts` sheet.

In [12]:
dec_sheet = (
    bc_concepts[['bc_id', 'dec_id', 'ncit_dec_code', 'dec_label', 'data_type', 'example_set']]
    .dropna(subset=['dec_id'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f'DataElementConcepts sheet: {dec_sheet.shape}')
print(f'BCs with at least one DEC: {dec_sheet["bc_id"].nunique()}')
dec_sheet.head()

DataElementConcepts sheet: (5472, 6)
BCs with at least one DEC: 1074


,bc_id,dec_id,ncit_dec_code,dec_label,data_type,example_set
0,C127127,C70713,C70713,Biospecimen Type,string,Urine
1,C127127,C48207,C48207,Unit of Concentration,string,NaN
2,C127127,C70856,C70856,Observation Result,decimal,NaN
3,C115789,C82525,C82525,Test Occurrence,string,N;Y
4,C115789,C82515,C82515,Collection Date Time,datetime,NaN


## 10c. Build `Categories` sheet

Controlled vocabulary of category tokens. Source: the xlsx `Categories` sheet (single column). This is the reference list the multi-value `BiomedicalConcept.categories` slot draws from.

In [13]:
categories_sheet = (
    categories_df[['category']]
    .dropna()
    .drop_duplicates()
    .sort_values('category')
    .reset_index(drop=True)
)

print(f'Categories sheet: {categories_sheet.shape}')
categories_sheet.head()

Categories sheet: (385, 1)


,category
0,6 Minute Walk Functional Test
1,6MWT
2,6MWT Functional Test Question
3,ADAS-Cog
4,ADAS-Cog CDISC Version Functional Test Question


## 10d. Build `BC_Categories` sheet

Normalised edge list — one row per `(bc_id, category)` pair. Split from the semicolon-delimited `bc_categories` field on the `BC Hierarchy` sheet. Parallels the SDTM-side `Relationships` sheet: the flat string stays on the BC sheet for locality, the edge list sits alongside for graph traversal.

In [14]:
# Split multi-value bc_categories into edges
bc_cat_raw = bc_hierarchy[['bc_id', 'bc_categories']].dropna(subset=['bc_categories'])

bc_cat_rows = []
for _, row in bc_cat_raw.iterrows():
    cats = [c.strip() for c in str(row['bc_categories']).split(';') if c.strip()]
    for cat in cats:
        bc_cat_rows.append({'bc_id': row['bc_id'], 'category': cat})

bc_categories_sheet = (
    pd.DataFrame(bc_cat_rows)
    .drop_duplicates()
    .reset_index(drop=True)
)

# Integrity check: every edge category should appear in the Categories vocab
edge_cats  = set(bc_categories_sheet['category'].unique())
vocab_cats = set(categories_sheet['category'].unique())
missing    = edge_cats - vocab_cats

print(f'BC_Categories sheet: {bc_categories_sheet.shape}')
print(f'BCs with at least one category: {bc_categories_sheet["bc_id"].nunique()}')
print(f'distinct categories used on edges: {len(edge_cats)}')
print(f'categories not in vocabulary sheet: {len(missing)}')
if missing:
    print('  sample:', sorted(missing)[:5])
bc_categories_sheet.head()

BC_Categories sheet: (4096, 2)
BCs with at least one category: 1345
distinct categories used on edges: 385
categories not in vocabulary sheet: 0


,bc_id,category
0,C127127,Laboratory Tests
1,C127127,Nitroso Carcinogens
2,C127127,Carcinogenic Nitrosamines
3,C115789,QRS
4,C115789,Functional Assessment


## 10e. Build `BC_Parents` sheet

BC-to-BC parent edges. One row per `(bc_id, parent_bc_id)` pair from the `BC Hierarchy` sheet, enriched with `parent_bc_short_name` for readability. Makes the BC hierarchy traversable without parsing the flat `bc_hierarchy_path` string.

Note: this is the BC-to-BC graph relation authored in the xlsx. It does not map to a LinkML slot on `BiomedicalConcept` (LinkML `parentConceptId` is the NCIt parent, a distinct relation).

In [15]:
bc_parents_sheet = (
    bc_hierarchy[['bc_id', 'parent_bc_id']]
    .dropna(subset=['parent_bc_id'])
    .drop_duplicates()
    .reset_index(drop=True)
)

# Enrich with parent's short_name
short_name_map = dict(zip(bc_hierarchy['bc_id'], bc_hierarchy['short_name']))
bc_parents_sheet['parent_bc_short_name'] = bc_parents_sheet['parent_bc_id'].map(short_name_map)

unresolved = bc_parents_sheet['parent_bc_short_name'].isna().sum()
print(f'BC_Parents sheet: {bc_parents_sheet.shape}')
print(f'BCs with parent: {bc_parents_sheet["bc_id"].nunique()}')
print(f'parent_bc_id values without a resolvable short_name: {unresolved}')
bc_parents_sheet.head()

BC_Parents sheet: (1073, 3)
BCs with parent: 1073
parent_bc_id values without a resolvable short_name: 0


,bc_id,parent_bc_id,parent_bc_short_name
0,C127127,C45398,Carcinogenic Nitrosamine
1,C115789,C81250,Functional Assessment
2,C115800,C115409,6MWT Functional Test Question
3,C115801,C115409,6MWT Functional Test Question
4,C115802,C115409,6MWT Functional Test Question


## 11. Anomaly audit

Counts for known issues. These go into the ReadMe; the flattener does not silently fix anything.


In [16]:
dss_with_edges = set(relationships_sheet['ds_id'].unique())
dss_all        = set(dss_sheet['ds_id'].unique())

issues = {
    'vlm_source_hyphen_rows':
        int(dss_df['vlm_source'].str.contains('-', na=False).sum()),
    'empty_reification_quad_rows':
        int((~rel_mask).sum()),
    'dss_without_any_edge':
        len(dss_all - dss_with_edges),
    'codelist_null_rows':
        int(variables_sheet['codelist_concept_id'].isna().sum()),
    'assigned_term_null_rows':
        int(variables_sheet['assigned_term_concept_id'].isna().sum()),
}
for k, v in issues.items():
    print(f'  {k:35s} : {v:>6,}')


  vlm_source_hyphen_rows              :     40
  empty_reification_quad_rows         :    313
  dss_without_any_edge                :      4
  codelist_null_rows                  :  5,830
  assigned_term_null_rows             :  8,683


## 12. Write `interim/COSMoS_Graph.xlsx`

Five data sheets + ReadMe. CT resolution happens in notebook 20, validation in notebook 30.


In [17]:
readme_lines = [
    'COSMoS Graph — schema-driven flatten of CDISC Library Excel exports',
    '',
    f'Generated : {date.today().isoformat()}',
    'Pipeline  : cosmos-graph/notebooks/10_flatten_schema_driven.ipynb',
    f'Input     : {DSS_XLSX.name}, {BC_XLSX.name}',
    'Schema    : cosmos-graph/reference/cosmos_linkml/cosmos_sdtm_model.yaml',
    '',
    'SHEETS',
    f'  DSS                  {len(dss_sheet):>6,} rows — one row per Dataset Specialization',
    f'  Variables            {len(variables_sheet):>6,} rows — one row per SDTMVariable (VLM-row grain)',
    f'  Relationships        {len(relationships_sheet):>6,} rows — reification quads',
    f'  Codelists            {len(codelists_sheet):>6,} rows — deduped codelist bindings with usage counts',
    f'  BC                   {len(bc_sheet):>6,} rows — one row per Biomedical Concept',
    f'  BC_Parents           {len(bc_parents_sheet):>6,} rows — BC-to-BC parent edges',
    f'  BC_Categories        {len(bc_categories_sheet):>6,} rows — BC-to-Category edges',
    f'  Categories           {len(categories_sheet):>6,} rows — category-token vocabulary',
    f'  Coding               {len(coding_sheet):>6,} rows — BC external code mappings (Coding inlined class)',
    f'  DataElementConcepts  {len(dec_sheet):>6,} rows — BC data element concepts (DEC inlined class)',
    '',
    'ANOMALIES',
]
for k, v in issues.items():
    readme_lines.append(f'  {k:35s} : {v:,}')
readme_lines += [
    '',
    'See cosmos-graph/docs/COSMoS_Graph.md for the graph model and design rationale.',
]

readme_df = pd.DataFrame({'README': readme_lines})

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name='ReadMe', index=False, header=False)
    dss_sheet.to_excel(writer, sheet_name='DSS', index=False)
    variables_sheet.to_excel(writer, sheet_name='Variables', index=False)
    relationships_sheet.to_excel(writer, sheet_name='Relationships', index=False)
    codelists_sheet.to_excel(writer, sheet_name='Codelists', index=False)
    bc_sheet.to_excel(writer, sheet_name='BC', index=False)
    bc_parents_sheet.to_excel(writer, sheet_name='BC_Parents', index=False)
    bc_categories_sheet.to_excel(writer, sheet_name='BC_Categories', index=False)
    categories_sheet.to_excel(writer, sheet_name='Categories', index=False)
    coding_sheet.to_excel(writer, sheet_name='Coding', index=False)
    dec_sheet.to_excel(writer, sheet_name='DataElementConcepts', index=False)

print(f'wrote {OUT_XLSX}')
print(f'size : {OUT_XLSX.stat().st_size:,} bytes')


wrote /sessions/gifted-eloquent-faraday/mnt/cdisc-for-ai/cosmos-graph/interim/COSMoS_Graph.xlsx
size : 2,038,763 bytes


## 13. Apply yellow layout

Re-open the written xlsx and apply the shared cosmos-track layout pattern: gold header, pale yellow data fill, thin borders, wrap-text cells, auto-widths. Interim file, but still a workbook we read for exploration.


In [18]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment

YELLOW_HEADER = PatternFill(start_color='FFD700', end_color='FFD700', fill_type='solid')
YELLOW_FILL   = PatternFill(start_color='FFFCE8', end_color='FFFCE8', fill_type='solid')
README_HEADER = PatternFill(start_color='595959', end_color='595959', fill_type='solid')
HEADER_FONT   = Font(bold=True)
HEADER_WHITE  = Font(bold=True, color='FFFFFF')
thin_border = Border(
    left=Side(style='thin', color='999999'),
    right=Side(style='thin', color='999999'),
    top=Side(style='thin', color='999999'),
    bottom=Side(style='thin', color='999999'),
)

def auto_width(ws, max_width=50):
    for col in ws.columns:
        width = max((len(str(c.value)) for c in col if c.value is not None), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(width + 2, max_width)

def format_data_sheet(ws):
    for cell in ws[1]:
        cell.font = HEADER_FONT
        cell.fill = YELLOW_HEADER
        cell.border = thin_border
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.fill = YELLOW_FILL
            cell.border = thin_border
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    auto_width(ws)

wb = load_workbook(OUT_XLSX)
for sheet_name in ('BC', 'BC_Parents', 'BC_Categories', 'Categories', 'Coding', 'DataElementConcepts', 'DSS', 'Variables', 'Relationships', 'Codelists'):
    format_data_sheet(wb[sheet_name])

# ReadMe — single-column, dark grey header band on row 1
ws = wb['ReadMe']
ws['A1'].font = HEADER_WHITE
ws['A1'].fill = README_HEADER
ws['A1'].alignment = Alignment(vertical='top')
ws.column_dimensions['A'].width = 80
wb.save(OUT_XLSX)
print(f're-saved with yellow layout: {OUT_XLSX}')


re-saved with yellow layout: /sessions/gifted-eloquent-faraday/mnt/cdisc-for-ai/cosmos-graph/interim/COSMoS_Graph.xlsx


## 14. Summary


In [19]:
xl = pd.ExcelFile(OUT_XLSX)
print(f'sheets in output: {xl.sheet_names}')
for s in xl.sheet_names:
    sh = pd.read_excel(OUT_XLSX, sheet_name=s, header=None if s == 'ReadMe' else 0)
    print(f'  {s:15s}: {sh.shape}')


sheets in output: ['ReadMe', 'DSS', 'Variables', 'Relationships', 'Codelists', 'BC', 'BC_Parents', 'BC_Categories', 'Categories', 'Coding', 'DataElementConcepts']
  ReadMe         : (27, 1)
  DSS            : (1326, 8)


  Variables      : (12677, 26)


  Relationships  : (12364, 6)
  Codelists      : (291, 3)
  BC             : (1345, 10)
  BC_Parents     : (1073, 3)


  BC_Categories  : (4096, 2)
  Categories     : (385, 1)
  Coding         : (105, 4)


  DataElementConcepts: (5472, 6)
